# Evaluasi Retrieval ChatKUHP dengan RAGAS

Notebook ini mengevaluasi performa sistem retrieval **ChatKUHP** menggunakan framework [RAGAS](https://docs.ragas.io/).

## Metrik yang Digunakan

| Metrik | Penjelasan |
|---|---|
| **Context Precision** | Seberapa presisi konteks yang diambil — apakah node yang diambil memang relevan dengan ground truth? |
| **Context Recall** | Seberapa lengkap konteks yang diambil — apakah semua informasi yang dibutuhkan ada dalam konteks yang diambil? |

> **Catatan:** Evaluasi ini hanya menguji komponen *retrieval* (endpoint `/getcontext`), bukan kualitas jawaban LLM. Endpoint `/getcontext` mengembalikan `selected_hierarchy` berupa pasal-pasal hukum hasil DFS yang dipilih sebagai konteks.

## Dependency

In [ ]:
# Jalankan sekali untuk instalasi
# !pip install ragas datasets pandas requests langchain-cohere langchain-google-genai

## Import & Configuration

In [ ]:
import os
import json
import requests
import pandas as pd
from typing import List, Dict
from datasets import Dataset as HFDataset

from ragas import evaluate
from ragas.metrics import context_precision, context_recall

from langchain_cohere import ChatCohere
from langchain_cohere import CohereEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

## Variable Configuration

In [ ]:
# ── Konfigurasi ──────────────────────────────────────────────────────
URL_SERVER    = "http://localhost:8000"
CSV_PATH      = "ground_truth.csv"
COHERE_API_KEY = os.environ.get("COHERE_API_KEY", "YOUR_COHERE_API_KEY")
REQUEST_TIMEOUT = 120

## Helper Function

### `get_context(url_server, question)`
Memanggil endpoint `/getcontext` pada backend untuk mendapatkan konteks hukum (pasal-pasal hasil DFS) dari sebuah pertanyaan.

Endpoint mengembalikan:
```json
{
  "chosen_goal": "KUHP Pasal 362",
  "goal_choices": [...],
  "contexts": { "KUHP Pasal 362": { "description": "...", ... }, ... }
}
```

RAGAS mengharapkan `contexts` berupa `List[str]`, sehingga kita akan mengekstrak teks deskripsi dari setiap node.

### `load_dataset(csv_path)`
Membaca CSV ground truth. CSV harus memiliki kolom:
- `question` — pertanyaan atau kasus hukum pengguna
- `ground_truth` — jawaban referensi yang benar

### `build_ragas_dataset(df, url_server)`
Membangun dataset dalam format yang dibutuhkan RAGAS dengan memanggil `/getcontext` untuk setiap baris data.

In [ ]:
def get_context(url_server: str, question: str) -> List[str]:
    """
    Memanggil endpoint /getcontext pada backend dan mengembalikan
    list teks konteks hukum (deskripsi pasal hasil DFS).
    """
    url = url_server.rstrip("/") + "/getcontext"
    payload = {"query": question}
    headers = {"Content-Type": "application/json"}

    resp = requests.post(url, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    data = resp.json()

    contexts_dict: dict = data.get("contexts", {})
    context_texts: List[str] = []
    for node_name, node_data in contexts_dict.items():
        desc = node_data.get("description", "")
        if desc:
            context_texts.append(f"{node_name}: {desc}")

    if not context_texts:
        chosen = data.get("chosen_goal", "")
        if chosen:
            context_texts = [chosen]

    return context_texts


def load_dataset(csv_path: str) -> pd.DataFrame:
    """
    Membaca CSV ground truth dan memvalidasi kolom yang diperlukan.
    """
    df = pd.read_csv(csv_path)
    if "question" not in df.columns:
        raise ValueError("CSV tidak memiliki kolom 'question'")
    if "ground_truth" not in df.columns:
        raise ValueError("CSV tidak memiliki kolom 'ground_truth'")
    return df


def build_ragas_dataset(df: pd.DataFrame, url_server: str) -> Dict[str, List]:
    """
    Membangun dataset RAGAS dari DataFrame.
    Untuk setiap baris, memanggil /getcontext untuk mendapatkan konteks retrieval.

    Format output yang dibutuhkan RAGAS:
    {
        "question"    : List[str]
        "answer"      : List[str]   -- kosong karena hanya eval retrieval
        "contexts"    : List[List[str]]
        "ground_truth": List[str]
    }
    """
    questions:    List[str]       = []
    answers:      List[str]       = []
    contexts:     List[List[str]] = []
    ground_truths: List[str]      = []

    total = len(df)
    for i, (_, row) in enumerate(df.iterrows()):
        q  = str(row["question"]).strip()
        gt = str(row.get("ground_truth", "")).strip()

        if not q:
            print(f"[SKIP] Baris {i+1}: pertanyaan kosong.")
            continue

        print(f"[{i+1}/{total}] Mengambil konteks untuk: {q[:80]}...")
        try:
            ctx = get_context(url_server, q)
        except Exception as e:
            print(f"  → ERROR: {e}. Konteks dikosongkan.")
            ctx = []

        questions.append(q)
        answers.append("")     # dikosongkan — hanya evaluasi retrieval
        contexts.append(ctx)
        ground_truths.append(gt)

    return {
        "question":    questions,
        "answer":      answers,
        "contexts":    contexts,
        "ground_truth": ground_truths,
    }

## Load Dataset & Build RAGAS Dataset

In [ ]:
df = load_dataset(CSV_PATH)
print(f"Total data ground truth: {len(df)} baris")
df.head()

In [ ]:
print("Membangun RAGAS dataset (memanggil /getcontext untuk setiap pertanyaan)...")
ragas_dict = build_ragas_dataset(df, URL_SERVER)

hf_ragas = HFDataset.from_dict(ragas_dict)
print(f"\nDataset berhasil dibuat: {hf_ragas}")

## Setup LLM Judge for RAGAS

In [ ]:
# LLM judge menggunakan Cohere
llm_judge = LangchainLLMWrapper(
    ChatCohere(
        model="command-a-plus-08-2025",
        cohere_api_key=COHERE_API_KEY,
        temperature=0.0,
    )
)

emb_judge = LangchainEmbeddingsWrapper(
    CohereEmbeddings(
        model="embed-multilingual-v3.0",
        cohere_api_key=COHERE_API_KEY,
    )
)

## RAGAS Evaluation

In [ ]:
print("Menjalankan evaluasi RAGAS...")

result = evaluate(
    dataset=hf_ragas,
    metrics=[context_precision, context_recall],
    llm=llm_judge,
    embeddings=emb_judge,
)

print("\n===== HASIL EVALUASI RAGAS =====")
print(result)

## Result as Dataframe

In [ ]:
result_df = result.to_pandas()
print("\nRingkasan Metrik:")
print(f"  Context Precision : {result_df['context_precision'].mean():.4f}")
print(f"  Context Recall    : {result_df['context_recall'].mean():.4f}")

result_df

In [ ]:
# Simpan hasil ke CSV (opsional)
output_path = "ragas_results.csv"
result_df.to_csv(output_path, index=False)
print(f"Hasil disimpan ke: {output_path}")